In [16]:
!pip install nltk scikit-learn pandas

In [17]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [18]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
from google.colab import files
uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [20]:
df = pd.read_csv("dataset.csv")  # replace with your file
df.head()

,review,sentiment
0,This movie was amazing and I loved it,positive
1,"Absolutely terrible film, waste of time",negative
2,Great acting and wonderful story,positive
3,"I hated this movie, it was boring",negative
4,"Best movie ever, highly recommend",positive


In [21]:
print("Shape:", df.shape)
print("Class Distribution:\n", df['sentiment'].value_counts())

print("\nSample Text:")
print(df['review'][0])

Shape: (20, 2)
Class Distribution:
 sentiment
positive    10
negative    10
Name: count, dtype: int64

Sample Text:
This movie was amazing and I loved it


In [22]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()  # lowercase

    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation/numbers

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords

    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # lemmatization

    return " ".join(tokens)

In [23]:
df['clean_text'] = df['review'].apply(clean_text)

In [24]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [26]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [27]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)

In [28]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)

In [29]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
dt_pred = dt.predict(X_test_tfidf)

In [30]:
def evaluate(y_true, y_pred, name):
    print(f"\n{name} Performance")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall:", recall_score(y_true, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))
    print(classification_report(y_true, y_pred))

In [31]:
evaluate(y_test, lr_pred, "Logistic Regression")
evaluate(y_test, nb_pred, "Naive Bayes")
evaluate(y_test, dt_pred, "Decision Tree")


Logistic Regression Performance
Accuracy: 0.5
Precision: 0.8333333333333334
Recall: 0.5
F1 Score: 0.5
              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
    positive       0.33      1.00      0.50         1

    accuracy                           0.50         4
   macro avg       0.67      0.67      0.50         4
weighted avg       0.83      0.50      0.50         4


Naive Bayes Performance
Accuracy: 0.75
Precision: 0.875
Recall: 0.75
F1 Score: 0.7666666666666667
              precision    recall  f1-score   support

    negative       1.00      0.67      0.80         3
    positive       0.50      1.00      0.67         1

    accuracy                           0.75         4
   macro avg       0.75      0.83      0.73         4
weighted avg       0.88      0.75      0.77         4


Decision Tree Performance
Accuracy: 0.25
Precision: 0.0625
Recall: 0.25
F1 Score: 0.1
              precision    recall  f1-score   support

  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [32]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, nb_pred),
        accuracy_score(y_test, dt_pred)
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression      0.50
1          Naive Bayes      0.75
2        Decision Tree      0.25
